# Final decoder runs, track 1: the run record

This notebook turns the two waves fetched from the Modal volume into the committed record under
`experiments/2026-11-25-decoder-final-track1/`. It trains nothing. For every job it copies the
probabilities and the run record, writes the single-pass and symmetrized dev predictions in the
official format, and scores them with the official scorer. It then writes the per-seed score
table, the training history, the measured costs and the repeat check against the preliminary run
of 14 November.

Run `01_lambda_selection.ipynb` first: this notebook refuses to run without the committed
`lambda_selection.json`, because dev scores may only be computed after the lambda choice is fixed.
Means over seeds, intervals and verdicts are computed elsewhere (`03_*`), from the files written here.

In [ ]:
TRACK = 1
RUN_FOLDER = "2026-11-25-decoder-final-track1"
SEEDS = (13, 42, 1234)
WAVES = {
    "A": {"run_id": "2026-11-25-decoder-final-wave-a", "app_id": "ap-eOvzhee5YGcEo56Mn1oenz", "jobs": "wave_a_jobs.json"},
    "B": {"run_id": "2026-11-26-decoder-final-wave-b", "app_id": "ap-lBQKpGhqBd5dga2exsuOeR", "jobs": "wave_b_jobs.json"},
}
SELECTION_COMMIT = "324debe2891c596f2cd6a57b24fe20311ebe4282"
SELECTION_COMMIT_TIME = "2026-11-26T10:02:27+05:00"
PRELIMINARY_FOLDER = "2026-11-14-decoder-preliminary-track1"
REPEATED = ("qwen2.5-1.5b-zero-shot", "qwen2.5-1.5b-lora-lambda0-seed13", "qwen2.5-1.5b-lora-lambda1-seed13")
LAUNCH = "PYTHONIOENCODING=utf-8 uv run modal run --detach scripts/modal/train_decoder.py::main --run-id {run_id} --jobs experiments/2026-11-25-decoder-final-track1/{jobs}"
COMMAND = "PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-3/02_final_runs_track1.ipynb"

In [ ]:
import json
import shutil
import subprocess

import numpy as np
import pandas as pd

from dico_nli.data import LABELS, REPO_ROOT, load_reference
from dico_nli.experiments import EXPERIMENTS_ROOT, write_config
from dico_nli.method.reversal import symmetrize
from dico_nli.scoring import score, write_predictions, write_reference

run_dir = EXPERIMENTS_ROOT / RUN_FOLDER
fetched = REPO_ROOT / "artifacts" / "decoder-final-track1"
selection = json.loads((run_dir / "lambda_selection.json").read_text(encoding="utf-8"))
tracked = subprocess.run(["git", "cat-file", "-e", f"{SELECTION_COMMIT}:experiments/{RUN_FOLDER}/lambda_selection.json"],
                         cwd=REPO_ROOT, capture_output=True)
assert tracked.returncode == 0, "lambda_selection.json is not in the selection commit"
chosen = selection["chosen_lambda"]
print("chosen lambda:", chosen)

# This notebook owns these folders and rewrites them; the selection files and job lists stay.
for name in ("predictions", "scores", "probabilities", "validation_probabilities", "validation_reference", "runs", "logs"):
    shutil.rmtree(run_dir / name, ignore_errors=True)
    (run_dir / name).mkdir()
tables_dir = run_dir / "tables"
dev = load_reference(TRACK, "dev")
gold = write_reference(fetched / "dev_reference.csv", dev)
columns = list(LABELS)

## Collect and score every job

Dev predictions are recomputed from the saved probabilities and must equal the prediction files
the job wrote. Each seed's validation reference is the same for every job of that seed, so it is
stored once per seed.

In [ ]:
records, score_rows = {}, []
for wave, info in WAVES.items():
    wave_dir = fetched / info["run_id"]
    jobs = json.loads((run_dir / info["jobs"]).read_text(encoding="utf-8"))
    folders = sorted(path for path in wave_dir.iterdir() if path.is_dir())
    assert len(folders) == len(jobs) == 16, (wave, len(folders), len(jobs))
    for folder in folders:
        name = folder.name
        record = json.loads((folder / "validation" / "run.json").read_text(encoding="utf-8"))
        assert record["system"] == name and name not in records
        records[name] = {**record, "wave": wave}
        shutil.copy(folder / "validation" / "run.json", run_dir / "runs" / f"{name}.json")
        for part, target in (("dev", "probabilities"), ("validation", "validation_probabilities")):
            for order in ("forward", "reverse"):
                shutil.copy(folder / part / f"probabilities_{order}.csv", run_dir / target / f"{name}__{order}.csv")
        reference = run_dir / "validation_reference" / f"seed{record['split_seed']}.csv"
        if reference.exists():
            assert reference.read_bytes() == (folder / "validation" / "reference.csv").read_bytes(), name
        else:
            shutil.copy(folder / "validation" / "reference.csv", reference)

        forward = pd.read_csv(folder / "dev" / "probabilities_forward.csv")
        reverse = pd.read_csv(folder / "dev" / "probabilities_reverse.csv")
        assert list(forward["instance_id"]) == list(reverse["instance_id"]) == [i.instance_id for i in dev]
        decoded = {
            "single": (forward[columns].to_numpy().argmax(axis=1), "predictions.csv"),
            "symmetrized": (symmetrize(forward[columns].to_numpy(), reverse[columns].to_numpy()).argmax(axis=1),
                            "predictions_symmetrized.csv"),
        }
        for decoding, (best, written_by_job) in decoded.items():
            predictions = {i: LABELS[k] for i, k in zip(forward["instance_id"], best)}
            from_job = pd.read_csv(folder / "dev" / written_by_job)
            assert predictions == dict(zip(from_job["instance_id"], from_job["label"])), (name, decoding)
            path = write_predictions(run_dir / "predictions" / f"{name}__{decoding}.csv", predictions)
            scores = score(gold, path, run_dir / "scores" / f"{name}__{decoding}")
            report = json.loads(scores.report_path.read_text(encoding="utf-8"))
            config = record["config"]
            score_rows.append({
                "system": name,
                "wave": wave,
                "model": config["model_name"],
                "mode": record["mode"],
                "lambda": config["consistency_weight"] if record["mode"] == "lora" else None,
                "seed": record["split_seed"] if record["mode"] == "lora" else None,
                "reverse_negatives": config["reverse_negatives"],
                "answer_format": config["answer_format"],
                "kept_epoch": record["best_epoch"],
                "decoding": decoding,
                "weighted_f1": scores.weighted_f1,
                "soft_cons": scores.soft_cons,
                "hard_cons": scores.hard_cons,
                "accuracy": report["classification"]["accuracy"],
                "macro_f1": report["classification"]["macro_f1"],
            })

assert len(records) == 32
source_digests = {record["code"]["source_sha256"] for record in records.values()}
assert len(source_digests) == 1, source_digests
assert all(record["code"]["repo_dirty"] is False for record in records.values())
assert all(record["mode"] == "zero-shot" or record["validation_check"]["difference"] == 0.0 for record in records.values())
dev_scores = pd.DataFrame(score_rows)
dev_scores.to_csv(tables_dir / "dev_scores_per_seed.csv", index=False)
print(len(dev_scores), "scored prediction files, one source digest:", next(iter(source_digests))[:12])
dev_scores.round(4)

## Training history and measured costs

Seconds, GPU and peak memory are what each job measured. The billed dollars are per wave, from
Modal's billing report saved in `billing_report.json`.

In [ ]:
history = pd.DataFrame([
    {"system": name, "wave": record["wave"], "kept_epoch": record["best_epoch"],
     **{key: value for key, value in row.items()}}
    for name, record in records.items() for row in record["history"]
])
history.to_csv(tables_dir / "training_history.csv", index=False)

costs = pd.DataFrame([
    {"system": name, "wave": record["wave"], "mode": record["mode"], "gpu": record["gpu"],
     "precision": record["precision"],
     "train_seconds": record["train_seconds"], "validation_seconds": record["validation_seconds"],
     "predict_seconds": record["predict_seconds"], "peak_gpu_memory_gb": record["peak_gpu_memory_gb"],
     "trainable_parameters": record["trainable_parameters"], "total_parameters": record["total_parameters"],
     "fit_size": record["fit_size"], "validation_size": record["holdout_size"], "dev_size": record["dev_size"]}
    for name, record in records.items()
])
costs.to_csv(tables_dir / "run_costs.csv", index=False)

billing = json.loads((run_dir / "billing_report.json").read_text(encoding="utf-8"))["apps"]
wave_rows = []
for wave, info in WAVES.items():
    part = costs[costs["wave"] == wave]
    billed = billing[info["app_id"]]["cost"]
    measured = float(part[["train_seconds", "validation_seconds", "predict_seconds"]].sum().sum())
    wave_rows.append({
        "wave": wave, "run_id": info["run_id"], "app_id": info["app_id"], "jobs": len(part),
        "measured_gpu_work_seconds": round(measured, 1),
        "billed_l4_usd": billed["L4"], "billed_cpu_usd": billed["CPU"], "billed_memory_usd": billed["Memory"],
        "billed_network_usd": billed["Network Egress"], "billed_total_usd": round(sum(billed.values()), 8),
    })
wave_costs = pd.DataFrame(wave_rows)
wave_costs.to_csv(tables_dir / "wave_costs.csv", index=False)
wave_costs

## Repeat check against the preliminary run

Wave A reran the zero-shot model and seed 13 at lambda 0 and lambda 1 with the same data, seed,
GPU type and settings as the preliminary run of 14 November. The table reports the largest
absolute difference between the saved dev probabilities and the share of instances with the
same predicted label.

In [ ]:
preliminary = EXPERIMENTS_ROOT / PRELIMINARY_FOLDER
repeat_rows = []
for name in REPEATED:
    row = {"system": name}
    now, before = {}, {}
    for order in ("forward", "reverse"):
        now[order] = pd.read_csv(run_dir / "probabilities" / f"{name}__{order}.csv")
        before[order] = pd.read_csv(preliminary / "probabilities" / f"{name}__{order}.csv")
        assert list(now[order]["instance_id"]) == list(before[order]["instance_id"])
        row[f"largest_probability_difference_{order}"] = float(
            np.abs(now[order][columns].to_numpy() - before[order][columns].to_numpy()).max())
    for decoding in ("single", "symmetrized"):
        new = pd.read_csv(run_dir / "predictions" / f"{name}__{decoding}.csv")
        old = pd.read_csv(preliminary / "predictions" / f"{name}__{decoding}.csv")
        assert list(new["instance_id"]) == list(old["instance_id"])
        row[f"label_agreement_{decoding}"] = float((new["label"] == old["label"]).mean())
        row[f"labels_changed_{decoding}"] = int((new["label"] != old["label"]).sum())
    old_record = json.loads((preliminary / "runs" / f"{name}.json").read_text(encoding="utf-8"))
    row["kept_epoch_preliminary"], row["kept_epoch_final"] = old_record["best_epoch"], records[name]["best_epoch"]
    repeat_rows.append(row)
repeat = pd.DataFrame(repeat_rows)
repeat.to_csv(tables_dir / "repeat_check.csv", index=False)
repeat

## Logs and the run configuration

In [ ]:
for info in WAVES.values():
    shutil.copy(fetched / f"{info['run_id']}.log", run_dir / "logs" / f"{info['run_id']}.log")

waves = {}
for wave, info in WAVES.items():
    commits = {record["code"]["repo_commit"] for record in records.values() if record["wave"] == wave}
    assert len(commits) == 1
    waves[wave] = {**info, "repo_commit": next(iter(commits)), "source_sha256": next(iter(source_digests)),
                   "command": LAUNCH.format(run_id=info["run_id"], jobs=info["jobs"])}
first = next(iter(records.values()))
write_config(run_dir, seed=13, track=TRACK, command=COMMAND, extra={
    "seeds": list(SEEDS),
    "zero_shot_validation_split_seed": 13,
    "chosen_lambda": chosen,
    "selection_commit": SELECTION_COMMIT,
    "selection_commit_time": SELECTION_COMMIT_TIME,
    "selection_notebook_command": selection["command"],
    "waves": waves,
    "gpu": sorted({record["gpu"] for record in records.values()}),
    "precision": first["precision"],
    "settings": {key: first["config"][key] for key in ("lora_rank", "lora_alpha", "lora_dropout", "target_modules",
                 "max_length", "batch_size", "micro_batch_size", "epochs", "learning_rate", "warmup_fraction", "weight_decay")},
    "holdout_fraction": first["holdout_fraction"],
    "versions": first["versions"],
    "fetch_command": "PYTHONIOENCODING=utf-8 uv run python scripts/modal/fetch_wave.py --run-id <run id> --jobs <jobs file> --part validation|dev --dest artifacts/decoder-final-track1",
    "volume": "dico-nli-runs",
})
print(json.dumps(waves, indent=2))